# Лабораторная 1: Transformer encoder для order-sensitive toy classification


## Что нужно знать до старта

Перед началом этой ЛР полезно открыть:
- [../README.md](./README.md)
- [guides/00_transformer_prerequisites.md](./guides/00_transformer_prerequisites.md)
- [guides/01_self_attention_and_positional_encoding_beginner.md](./guides/01_self_attention_and_positional_encoding_beginner.md)
- [guides/02_transformer_encoder_toy_walkthrough.md](./guides/02_transformer_encoder_toy_walkthrough.md)
- [../../02-Attention/theory/theory.md](../../02-Attention/theory/theory.md)

Это первая лабораторная блока `03-Transformer` и Шаг 5 общего курса.


## Интуиция задачи без формул

Здесь модель решает очень специальную задачу:
- во входной последовательности точно есть токены `7` и `3`;
- метка зависит не от того, что эти токены вообще были,
- а от того, **кто встретился раньше**.

Пример:

```text
[7, 5, 2, 3, PAD, PAD, ...] -> y = 1
[3, 5, 2, 7, PAD, PAD, ...] -> y = 0
```

Набор токенов одинаковый, но порядок разный.
Это и есть главная причина, по которой Transformer encoder не может обходиться без positional embedding.


## Как проходить эту ЛР без преподавателя

Фиксированный порядок для темы `03-Transformer`:
1. Прочитать `guides/00_transformer_prerequisites.md`.
2. Прочитать `guides/01_self_attention_and_positional_encoding_beginner.md`.
3. Пройти `guides/02_transformer_encoder_toy_walkthrough.md`.
4. Сделать свою первую попытку в этом starter notebook.
5. Если застряли, открыть `guides/04_transformer_debugging_playbook.md`.
6. Сделать вторую попытку после debugging-step.
7. Только потом смотреть в `solutions/01_transformer_encoder_order_toy_solution.ipynb`.


## Что изменилось после `02-Attention / ЛР01`

В attention-лаборатории decoder смотрел на encoder.

Теперь:
- рекуррентной ячейки больше нет;
- одна и та же последовательность сама строит `query`, `key`, `value`;
- вместо `cross-attention` мы используем `self-attention`;
- positional embedding становится обязательной частью модели.


## Контракт данных

Вход:
- padded последовательности целых token ids,
- форма `X -> (N, T)`.

Выход:
- бинарная метка `y -> (N,)`.

Правило метки:
- `y = 1`, если `7` стоит раньше `3`;
- `y = 0`, если `3` стоит раньше `7`.


## Таблица форм тензоров

| Сущность | Смысл | Форма |
|---|---|---|
| `X_train` | padded token ids | `(N, T)` |
| `padding_mask` | полезные позиции | `(N, T)` |
| `embeddings` | token + position embeddings | `(N, T, E)` |
| `attention_scores` | веса внимания по головам | `(N, H, T, T)` |
| `pooled` | один вектор на объект | `(N, E)` |
| `y_pred` | вероятность класса | `(N, 1)` |


## Шпаргалка по обозначениям и формам

- `N` — число объектов.
- `T` — длина последовательности после padding.
- `E` — размер embedding / model dimension.
- `H` — число attention heads.
- `PAD = 0`.

Практический фокус этой ЛР:
- проверить shapes;
- проверить mask;
- проверить, что attention не уходит в padded хвост.


## Контракт модели

Модель должна состоять из таких блоков:
1. `TokenAndPositionEmbedding`
2. `TransformerEncoderBlock`
3. masked average pooling
4. classifier head для binary classification

Отдельно нужен tracing-path, который возвращает `attention_scores` хотя бы для одного encoder block.


## Мини-теория

Минимальный encoder block:

$$
H_1 = \mathrm{LayerNorm}(X + \mathrm{MHA}(X))
$$

$$
H_2 = \mathrm{LayerNorm}(H_1 + \mathrm{FFN}(H_1))
$$

Если positional embedding отсутствует, то attention знает “какие токены были”, но слабо знает “где они были”.
Для этой лабораторной это критично, потому что метка зависит именно от порядка.


## Ручной разбор одного примера

Сравните две последовательности:

```text
A = [7, 5, 2, 3]
B = [3, 5, 2, 7]
```

Если смотреть только на набор токенов, они одинаковы.
Но по правилу задачи:
- `A -> 1`
- `B -> 0`

Значит, модели нужен позиционный сигнал.


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


In [ ]:
SEED = 7
PAD_ID = 0
KEY_TOKEN = 7
VALUE_TOKEN = 3
VOCAB_SIZE = 16
SEQ_LEN = 12
MIN_LEN = 4
TRAIN_SIZE = 4000
TEST_SIZE = 1000
EMBED_DIM = 32
NUM_HEADS = 2
FF_DIM = 64
BATCH_SIZE = 64
EPOCHS = 6

keras.utils.set_random_seed(SEED)
np.set_printoptions(linewidth=120)


## Checkpoint 1: данные

На этом этапе нужно убедиться в трёх вещах:
- padded последовательности имеют общую длину `T`;
- токены `7` и `3` всегда присутствуют в полезной части;
- label зависит именно от порядка этих токенов.

Двигайтесь дальше только когда shape и rule label уже понятны руками.


In [ ]:
filler_tokens = np.array(
    [token for token in range(1, VOCAB_SIZE) if token not in (KEY_TOKEN, VALUE_TOKEN)],
    dtype=np.int32,
)

def generate_order_dataset(num_samples, seq_len=SEQ_LEN, min_len=MIN_LEN, seed=SEED):
    rng = np.random.default_rng(seed)
    X = np.full((num_samples, seq_len), PAD_ID, dtype=np.int32)
    y = np.zeros((num_samples,), dtype=np.int32)
    lengths = np.zeros((num_samples,), dtype=np.int32)

    for i in range(num_samples):
        # TODO 1.1: выберите полезную длину в диапазоне [min_len, seq_len]
        # TODO 1.2: соберите последовательность filler-токенов и вставьте KEY_TOKEN и VALUE_TOKEN
        # TODO 1.3: посчитайте label по порядку этих токенов и запишите padded sequence в X
        raise NotImplementedError('TODO 1: implement dataset generation')

    return X, y, lengths

X_all, y_all, lengths_all = generate_order_dataset(TRAIN_SIZE + TEST_SIZE)
X_train, X_test, y_train, y_test, len_train, len_test = train_test_split(
    X_all,
    y_all,
    lengths_all,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y_all,
)

print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)


In [ ]:
manual_a = np.array([[7, 5, 2, 3, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=np.int32)
manual_b = np.array([[3, 5, 2, 7, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=np.int32)

print('manual A label should be 1')
print('manual B label should be 0')
print('useful lengths sample:', len_train[:5])
print('class balance      :', np.bincount(y_train))


## Checkpoint 2: embeddings + mask

Здесь нужно проверить две базовые идеи:
- `TokenAndPositionEmbedding` возвращает `(batch, time, embed_dim)`;
- padding mask сохраняется и потом попадёт в attention и pooling.

Если здесь shape или mask ведут себя странно, дальше в модели будет только больше шума.


In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.pos_emb = layers.Embedding(maxlen, embed_dim)
        self.supports_masking = True

    def call(self, inputs):
        # TODO 2.1: построить позиции через tf.range
        # TODO 2.2: получить token embeddings и position embeddings
        # TODO 2.3: вернуть их сумму
        raise NotImplementedError('TODO 2: implement TokenAndPositionEmbedding.call')

    def compute_mask(self, inputs, mask=None):
        # TODO 2.4: вернуть mask от token embedding
        raise NotImplementedError('TODO 2: forward token mask')


def masked_average(x, mask):
    mask = tf.cast(mask, x.dtype)
    mask = tf.expand_dims(mask, axis=-1)
    summed = tf.reduce_sum(x * mask, axis=1)
    counts = tf.reduce_sum(mask, axis=1)
    return summed / tf.maximum(counts, 1.0)


class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=rate,
        )
        self.ffn = keras.Sequential(
            [
                layers.Dense(ff_dim, activation='relu'),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)
        self.supports_masking = True

    def call(self, inputs, mask=None, training=None, return_attention_scores=False):
        # TODO 3.1: собрать padding-aware attention_mask из mask
        # TODO 3.2: вызвать self.att(..., return_attention_scores=...)
        # TODO 3.3: добавить residual + LayerNorm + FFN + residual + LayerNorm
        raise NotImplementedError('TODO 3: implement TransformerEncoderBlock.call')

    def compute_mask(self, inputs, mask=None):
        return mask


In [ ]:
sample_tokens = X_train[:2]
sample_mask = sample_tokens != PAD_ID

sample_embedding_layer = TokenAndPositionEmbedding(SEQ_LEN, VOCAB_SIZE, EMBED_DIM)
sample_encoder_block = TransformerEncoderBlock(EMBED_DIM, NUM_HEADS, FF_DIM)

sample_embeddings = sample_embedding_layer(sample_tokens)
sample_encoded, sample_scores = sample_encoder_block(
    sample_embeddings,
    mask=sample_mask,
    return_attention_scores=True,
)

print('sample_embeddings:', sample_embeddings.shape)
print('sample_encoded   :', sample_encoded.shape)
print('sample_scores    :', sample_scores.shape)


## Checkpoint 3: encoder block + classifier

После этого блока модель должна:
- принимать `tokens -> (batch, time)`;
- превращать их в embeddings с позициями;
- прогонять через `TransformerEncoderBlock`;
- сворачивать всё в одну вероятность `y_pred -> (batch, 1)` через `sigmoid`.


In [ ]:
keras.utils.set_random_seed(SEED)

transformer_inputs = keras.Input(shape=(SEQ_LEN,), dtype='int32', name='tokens')
padding_mask = layers.Lambda(lambda x: tf.not_equal(x, PAD_ID), name='padding_mask')(transformer_inputs)

# TODO 4.1: создать TokenAndPositionEmbedding и TransformerEncoderBlock
# TODO 4.2: прогнать inputs через embedding и encoder block
# TODO 4.3: сделать masked average pooling, небольшой Dense classifier head и финальный sigmoid
# TODO 4.4: скомпилировать model
raise NotImplementedError('TODO 4: build the transformer classifier model')


In [ ]:
model.summary()


## Checkpoint 4: обучение

Перед запуском `fit` проверьте:
- classifier head действительно бинарный;
- loss = `binary_crossentropy`;
- train и validation будут сравниваться отдельно от финального test.


In [ ]:
# TODO 5.1: обучить model.fit(...) с validation_split
# TODO 5.2: сохранить результат в history
raise NotImplementedError('TODO 5: train the model')


In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()


## Checkpoint 5: attention trace и критерии завершения

Перед сдачей здесь должны одновременно выполняться все условия:
- `test_acc >= 0.95`;
- два ручных примера с перестановкой `7` и `3` дают разные предсказания;
- heatmap строится только по non-PAD части последовательности.

Сначала снимите итоговую метрику и ручные примеры, потом интерпретируйте attention trace.


In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'test_loss = {test_loss:.4f}')
print(f'test_acc  = {test_acc:.4f}')


In [ ]:
paired_examples = np.array(
    [
        [7, 5, 2, 3, 0, 0, 0, 0, 0, 0, 0, 0],
        [3, 5, 2, 7, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    dtype=np.int32,
)

paired_probs = model.predict(paired_examples, verbose=0).ravel()
for seq, prob in zip(paired_examples, paired_probs):
    label = int(prob >= 0.5)
    print(seq, '-> prob=', round(float(prob), 4), 'label=', label)


In [ ]:
# TODO 6.1: взять один тестовый пример
# TODO 6.2: получить attention_scores через tracing-path/model
# TODO 6.3: усреднить attention по головам и обрезать padded хвост
raise NotImplementedError('TODO 6: trace attention scores')


In [ ]:
token_labels = [str(token) for token in sample_tokens[0][:sample_length]]

plt.figure(figsize=(6, 5))
plt.imshow(mean_attention, cmap='magma', aspect='auto')
plt.colorbar(label='attention weight')
plt.xticks(range(sample_length), token_labels)
plt.yticks(range(sample_length), token_labels)
plt.xlabel('key positions')
plt.ylabel('query positions')
plt.title('Mean self-attention over heads')
plt.tight_layout()
plt.show()


## Опционально после сдачи: почему здесь не хватает bag-of-words логики

## Если не получилось с первого раза

Не начинайте с гипотезы “Transformer слишком сложный”.
Сначала проверьте:
- rule label;
- shapes;
- mask;
- позиционное кодирование;
- форму `attention_scores`.


## Если застряли: порядок диагностики

1. Проверить два ручных примера с одинаковым набором токенов.
2. Проверить форму `embeddings`.
3. Проверить форму `attention_scores`.
4. Проверить, что attention не смотрит на `PAD`.
5. Только потом смотреть в solution notebook.


## Чек-лист перед сдачей

- Данные padded до общей длины `T`.
- `7` и `3` всегда есть в полезной части последовательности.
- `TokenAndPositionEmbedding` возвращает `(batch, time, embed_dim)`.
- `TransformerEncoderBlock` использует mask.
- Модель возвращает бинарную вероятность `y_pred -> (N, 1)` через `sigmoid`.
- `test_acc >= 0.95`.
- Два ручных примера с перестановкой `7` и `3` различаются по предсказанию.
- Attention trace визуализируется хотя бы на одном примере и обрезан до non-PAD части.


## Как использовать решение без самообмана

Правильный порядок:
1. первая самостоятельная попытка;
2. walkthrough;
3. debugging playbook;
4. вторая самостоятельная попытка;
5. только потом solution notebook.

Если просто скопировать custom layer, то можно получить зелёные ячейки без понимания, зачем нужны positional embedding и padding mask.


## Опционально после сдачи: мини-экзамен

1. Почему self-attention без позиции не решает эту задачу надёжно?
2. Зачем здесь нужна padding mask, если это не `seq2seq`?
3. Почему `attention_scores` имеют две оси времени?
4. Что именно делает residual connection в encoder block?


## Опционально после сдачи: что дальше

Эта лабораторная закрывает synthetic-вход в `Transformer encoder`.

Следующий шаг курса — `03-Transformer / ЛР02`, где тот же блок переносится на реальный `IMDB`:
- вход уже не synthetic sequence, а review;
- метка остаётся одна на последовательность;
- self-attention и positional embedding остаются центральными.

Открывать дальше: [02_transformer_encoder_imdb.ipynb](./02_transformer_encoder_imdb.ipynb)


## Опционально после сдачи: вопросы для самопроверки

## Типичные ошибки (симптом -> причина -> исправление)

- `attention_scores` неожиданной формы -> перепутано ожидание выхода `MultiHeadAttention` -> помнить про `(batch, heads, T, T)`.
- Accuracy около `0.5` -> метка считается не по позиции, а по наличию токенов -> проверить rule label.
- Heatmap яркий на padded хвосте -> mask не передан в attention -> проверить `padding_mask`.
- Порядок не влияет на модель -> забыли positional embedding -> проверить `TokenAndPositionEmbedding`.
